# DC Motor Predictive Maintenance — Sequence-Based Model (Rolling Window)

**What's different from the basic model?**

The basic model looks at *one* sensor reading and classifies it. That's fault *detection* — reacting after the problem is already there.

This notebook looks at the *last 10 readings in a row* (a rolling window). It learns to recognise rising trends — like temperature slowly climbing over 2.5 minutes — and predicts what fault level is *coming*, before the sensor crosses a hard threshold. That's true **predictive** maintenance.

This is a meaningful upgrade for your project report and GitHub repo.


In [ ]:
# Step 1: Install / import everything needed
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries ready.")

## Step 2: Upload your dataset

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
df = pd.read_csv(csv_filename)

print(f"Loaded {len(df)} rows across {df['session_id'].nunique()} sessions.")
df.head()

## Step 3: Build rolling windows

Each training sample will be a window of 10 consecutive readings **from the same session**.
The label for each window is the fault level of the **last reading in that window** — so the model is always predicting the current state from the recent past.

**Window size = 10** means it looks at the last 150 seconds of sensor history (10 readings × 15 seconds each).


In [ ]:
WINDOW_SIZE = 10

feature_columns = [
    'accel_x_g', 'accel_y_g', 'accel_z_g',
    'gyro_x_dps', 'gyro_y_dps', 'gyro_z_dps',
    'vibration_rms_g', 'temperature_c'
]

X_windows = []
y_labels  = []

for session_id, group in df.groupby('session_id'):
    group = group.sort_values('step_in_session').reset_index(drop=True)
    data  = group[feature_columns].values
    labels = group['fault_name'].values

    for i in range(WINDOW_SIZE, len(group)):
        window = data[i - WINDOW_SIZE : i]   # 10 rows × 8 features
        X_windows.append(window.flatten())    # flatten to 80 features
        y_labels.append(labels[i])            # label = current fault level

X = np.array(X_windows)
y = np.array(y_labels)

print(f"Total windows: {len(X)}")
print(f"Each window has {X.shape[1]} features ({WINDOW_SIZE} steps × {len(feature_columns)} sensors)")
print(f"\nLabel distribution:")
print(pd.Series(y).value_counts())

## Step 4: Train / test split and model training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training windows: {len(X_train)}")
print(f"Testing windows:  {len(X_test)}")

model_seq = RandomForestClassifier(
    n_estimators=200,
    max_depth=14,
    random_state=42,
    class_weight='balanced'
)

model_seq.fit(X_train, y_train)
print("\nSequence model trained!")

## Step 5: Evaluate

In [ ]:
y_pred = model_seq.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Sequence Model Accuracy: {accuracy*100:.2f}%\n")
print(classification_report(y_test, y_pred))

In [ ]:
order = ['NORMAL', 'WATCH', 'WARNING', 'CRITICAL', 'FAILURE']
cm = confusion_matrix(y_test, y_pred, labels=order)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=order, yticklabels=order)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Sequence Model (10-reading window)')
plt.show()

## Step 6: Visualise a degrading session

This is the most powerful chart for your report — it shows a session where the motor is degrading, and compares:
- **Actual** fault level (what really happened, according to thresholds)
- **Predicted** fault level (what the sequence model said, reading by reading)

A good model will start raising the alarm *before* the actual threshold is crossed.


In [ ]:
# Pick a session that reached at least WARNING level
degrading_sessions = df[df['fault_name'].isin(['WARNING', 'CRITICAL', 'FAILURE'])]['session_id'].unique()

if len(degrading_sessions) == 0:
    print("No degrading sessions found.")
else:
    session_id = degrading_sessions[0]
    group = df[df['session_id'] == session_id].sort_values('step_in_session').reset_index(drop=True)
    data  = group[feature_columns].values
    actual_labels = group['fault_name'].values

    FAULT_ORDER = {'NORMAL': 0, 'WATCH': 1, 'WARNING': 2, 'CRITICAL': 3, 'FAILURE': 4}

    actual_scores = []
    predicted_scores = []
    steps = []

    for i in range(WINDOW_SIZE, len(group)):
        window = data[i - WINDOW_SIZE : i].flatten().reshape(1, -1)
        pred = model_seq.predict(window)[0]
        actual_scores.append(FAULT_ORDER[actual_labels[i]])
        predicted_scores.append(FAULT_ORDER[pred])
        steps.append(i)

    plt.figure(figsize=(12, 4))
    plt.plot(steps, actual_scores,    label='Actual fault level',    linewidth=2, color='steelblue')
    plt.plot(steps, predicted_scores, label='Predicted fault level', linewidth=2, color='tomato', linestyle='--')
    plt.yticks([0,1,2,3,4], ['NORMAL','WATCH','WARNING','CRITICAL','FAILURE'])
    plt.xlabel('Reading number in session')
    plt.ylabel('Fault level')
    plt.title(f'Session {session_id}: Actual vs Predicted fault level over time')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"Session {session_id} peak actual fault: {actual_labels[WINDOW_SIZE:][np.argmax(actual_scores)]}")

## Step 7: Save and download the sequence model

In [ ]:
import joblib

joblib.dump(model_seq, 'dc_motor_sequence_model.pkl')
files.download('dc_motor_sequence_model.pkl')

print("Sequence model saved as dc_motor_sequence_model.pkl")

## Summary

You now have two models:

| Model | What it sees | Use case |
|---|---|---|
| Basic model (`dc_motor_fault_model.pkl`) | 1 reading | Real-time fault classification |
| Sequence model (`dc_motor_sequence_model.pkl`) | Last 10 readings | Trend-based prediction (true predictive maintenance) |

For your report, the **Actual vs Predicted chart** (Step 6) is a strong figure — it visually demonstrates the model learning to anticipate fault escalation rather than just reacting to it.
